In [ ]:
import pandas as pd
import requests
from datetime import datetime, date, timedelta
from key.apikey import NEWS_API


date1 = (date.today() - timedelta(days=7)).isoformat() # Дата начала поиска
date2 = date.today().isoformat() # Текущая дата

url = "https://newsapi.org/v2/everything"

# Задаем слово для поиска в новостях
query = "Гренландия"

# Параметры поиска
params = {
    "q": query,
    "from": date1,
    "to": date2,
    "sortBy": "popularity",
    "language": "ru",
    "apiKey": NEWS_API
}

# Отсылаем get request запрос на сервер с заданными ранее параметрами
response = requests.get(url, params)

# Проверяем ошибки
response.raise_for_status()

# Передаем ответ от сервера в формате json
data = response.json()

# Генерируем имя файла
filename = f"news_{query}_{date1}_{date2}.csv"

df = pd.json_normalize(data['articles'])

df.insert(6,"upload_date", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# print("Тип столбца author:", df['author'].dtype)
# print("Уникальные значения:", df['author'].unique())
# print("Пропущенные значения:", df['author'].isna().sum())

# print("Примеры значений:")
# for i, val in enumerate(df['author'].head()):
#     print(f"  {i}: {repr(val)} (тип: {type(val)})")

# Замена пустых строк в колонке
df['author'] = df['author'].fillna('Неизвестно').replace('', 'Неизвестно')

# Удалил колонку, т.к. в ней дубли из колонки source.name и много пустых значений
del df['source.id']

# Преобразуем строку даты в datetime
df['publishedAt'] = pd.to_datetime(df['publishedAt'], format='%Y-%m-%dT%H:%M:%SZ')

# Сортируем по дате публикации
df = df.sort_values(by='publishedAt',ascending=False)

if len(df) > 0:
    df.to_csv(filename, sep=';',index=False)